## prepare


In [1]:
import numpy as np
np.float_ = np.float64
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch.utils.data import TensorDataset
import datasets
from datasets import load_from_disk, Image
import torch
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from torchmetrics.image.fid import FrechetInceptionDistance
import math
import copy

In [2]:
import os
import random
import numpy as np
import torch


def set_seed(seed: int):
    # 1. Python random
    random.seed(seed)

    # 2. NumPy
    np.random.seed(seed)

    # 3. PyTorch (CPU & GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # 如果你用多个 GPU

    # 4. cuDNN 设置为确定性模式，关闭 benchmark
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True





def seed_worker(worker_id):
    # 每个 worker 基于相同或偏移后的种子
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


# 在脚本最上方调用一次
SEED = 42
set_seed(SEED)

# PyTorch 1.7+ 可直接用 generator
gen = torch.Generator()
gen.manual_seed(SEED)

### dataset

In [3]:
cuda_num = 2
s = 32
p = 3
INPUT_DIM = p * s * s
LATENT_DIM = 20

HIDDEN_DIM = LATENT_DIM * 10
data_folder = "data_cifar100"
datasetname = "cif100"

# train+label
train_dataset = datasets.load_from_disk(f"/home/yzm/{data_folder}/train")
print(f"Total samples: {len(train_dataset)}")

train_images = (
    torch.stack([torch.tensor(np.array(img)) for img in train_dataset["img"]]).float()
    / 255.0
)
# CIFAR100: [N, 32, 32, 3] → 转成 [N, 3, 32, 32]
train_images = train_images.permute(0, 3, 1, 2)
train_images = train_images.flatten(start_dim=1)  # [N, 3072]

train_labels = torch.tensor(train_dataset["fine_label"], dtype=torch.long)
train_dataset_torch = TensorDataset(train_images, train_labels)


# test+label
test_dataset = datasets.load_from_disk(f"/home/yzm/{data_folder}/test")
print(f"Total samples: {len(test_dataset)}")

test_images = (
    torch.stack([torch.tensor(np.array(img)) for img in test_dataset["img"]]).float()
    / 255.0
)
test_images = test_images.permute(0, 3, 1, 2)
test_images = test_images.flatten(start_dim=1)   # [N, 3072]

test_labels = torch.tensor(test_dataset["fine_label"], dtype=torch.long)
test_dataset_torch = TensorDataset(test_images, test_labels)

Total samples: 50000
Total samples: 10000


In [4]:
from torch.utils.data import random_split

# 定义划分比例
train_size = int(0.9 * len(train_dataset_torch))  # 90%
val_size = len(train_dataset_torch) - train_size  # 10%

# 按比例划分数据集
train_dataset_torch, val_dataset_torch = random_split(train_dataset_torch, [train_size, val_size], generator=gen)

# 打印划分后的数据集大小
print(f"Train dataset size: {len(train_dataset_torch)}")
print(f"Validation dataset size: {len(val_dataset_torch)}")

Train dataset size: 45000
Validation dataset size: 5000


### vae class


In [5]:
class VAE_Encoder(nn.Module):
    """
    VAE的encoder，将输入的图片映射到隐变量的分布，输出隐变量分布的均值和方差.
    由于方差不能为负，所以这里输出的是log(方差)，而不是方差本身.
    """

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE_Encoder, self).__init__()

        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.Tanh())

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)


class VAE_Decoder(nn.Module):
    """
    VAE的decoder，将隐变量映射回图片.
    """

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE_Decoder, self).__init__()

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.decoder(z)


class VAE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM):
        super(VAE, self).__init__()

        self.latent_dim = latent_dim

        self.encoder = VAE_Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = VAE_Decoder(input_dim, hidden_dim, latent_dim)

    def encode(self, x):
        """
        将输入的图片映射到隐变量的分布，输出隐变量分布的均值和log(方差).
        """
        return self.encoder(x)

    def decode(self, z):
        """
        将隐变量映射回图片.
        """
        return self.decoder(z)

    def reparameterize(self, mu, logvar):
        """
        重参数化技巧，先从标准正态分布中采样一个epsilon，然后根据隐变量分布的均值和方差，计算出隐变量.
        """
        std = torch.exp(
            0.5 * logvar
        )  # 计算标准差, std = sqrt(var) = sqrt(exp(logvar)) = exp(logvar/2)
        epsilon = torch.randn_like(
            std, requires_grad=False
        )  # 从标准正态分布中采样epsilon
        return mu + epsilon * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

### loss


In [6]:
def loss_iwae_gauss(mu, logvar, x, decoder, k=5, sigma2=1.0, device=None):
    if device is None:
        device = x.device

    B = x.size(0)
    D = x.view(B, -1).size(1)
    z_dim = mu.size(-1)

    # 采样 k 个 z: shape [B, k, Z]
    eps = torch.randn(B, k, z_dim, device=device)
    std = torch.exp(0.5 * logvar)                          # [B, Z]
    z = mu.unsqueeze(1) + std.unsqueeze(1) * eps           # [B, k, Z]

    # 常量
    dtype = z.dtype
    log_2pi = torch.tensor(math.log(2 * math.pi), device=device, dtype=dtype)
    log_k = torch.tensor(math.log(k), device=device, dtype=dtype)

    # 第二项 log p(z) 标准正态先验: [B, k]
    log_pz = -0.5 * (z**2 + log_2pi).sum(-1)

    # 第三项 log q(z|x): [B, k]
    log_qz = -0.5 * (
        ((z - mu.unsqueeze(1)) ** 2) / (std.unsqueeze(1) ** 2)
        + logvar.unsqueeze(1)
        + log_2pi
    ).sum(-1)

    # 解码: flatten z -> [B*k, Z] and decode -> [B*k, D]
    z_flat = z.view(B * k, z_dim)
    x_recon = decoder(z_flat)                                # [B*k, D]
    x_expand = x.unsqueeze(1).expand(-1, k, -1).contiguous().view(B * k, -1)
    diff = x_expand - x_recon                                # [B*k, D]

    # 第一项 log p(x|z) 似然，固定方差 sigma2: [B, k]
    const_px = torch.tensor(0.5 * D * math.log(2 * math.pi * sigma2), device=device, dtype=diff.dtype)
    log_px = (-0.5 * (diff**2 / sigma2).sum(-1) - const_px).view(B, k)

    # importance weights: log-sum-exp 聚合 （per data point: [B]）
    log_w = log_px + log_pz - log_qz                        # [B, k]
    m = log_w.max(dim=1, keepdim=True).values               # [B, 1]
    sum_exp = torch.exp(log_w - m).sum(dim=1)               # [B]
    Lk = m.squeeze(1) + torch.log(sum_exp) - log_k          # [B]

    total_loss = - Lk.sum()
    return total_loss


In [7]:
def loss_function(x, recon_x, mu, logvar, prior_var=1.0, sigma2=1.0):
    """
    修改版: 支持动态调整先验方差 prior_var
    prior_var: scalar, 先验分布的方差 (默认1.0)
    """

    # 2. KL 散度 (General Gaussian Prior)
    # log(prior_var)
    log_prior_var = math.log(prior_var)
    
    # 公式推导转代码:
    # 0.5 * sum(log(sigma_p^2) - log(sigma^2) - 1 + (sigma^2 + mu^2)/sigma_p^2)
    # 提取 -0.5: -0.5 * sum(1 + log(sigma^2) - log(sigma_p^2) - (sigma^2 + mu^2)/sigma_p^2)
    
    var = logvar.exp()
    KLD = -0.5 * torch.sum(1 + logvar - log_prior_var - (mu.pow(2) + var) / prior_var)
    
    return KLD

In [8]:
def score_fn_torch(z: torch.Tensor, x_target: torch.Tensor, decoder: nn.Module, sigma2=1.0, prior_var=1.0) -> torch.Tensor:
    """
    修改版: 引入 prior_var 参数
    计算 ∇_z log p(z, x_target)
    """
    assert z.requires_grad, "z must have requires_grad=True"

    B, D = z.shape[0], z.shape[1]
    if x_target.ndim == 1:
        x_target = x_target.unsqueeze(0)
    
    x_target = x_target.view(B, -1)   # [B, 3072]
    x_rec = decoder(z).view(B, -1)
    diff = x_target - x_rec

    # --- 修改点 1: log p(z) 考虑先验方差 ---
    # log p(z) = -0.5 * (z^2 / prior_var)
    logpz = -0.5 * (z ** 2 / prior_var).sum(dim=1)

    # log p(x|z) = -1/(2σ²) * ||x - decoder(z)||²
    logpx = -0.5 / sigma2 * (diff ** 2).sum(dim=1)

    # --- 分别求两项梯度 ---
    # --- 修改点 2: grad_pz 除以先验方差 ---
    grad_pz = -z / prior_var  # [B, D]
    
    grad_px = torch.autograd.grad(logpx.sum(), z, create_graph=True, retain_graph=True)[0]

    # 合并两项得到 g(z)
    g = grad_pz + grad_px  # [B, D]
    
    return g

In [9]:
def bamloss_batch(
    mu, logvar, x_target, decoder,
    sample_K=2, lambda_t=0.1, sigma2=1.0, prior_var=1.0, # <--- 新增参数
):

    N, D = mu.shape

    # === 1. 重参数化采样 z (这就是你报错缺失的部分) ===
    std = torch.exp(0.5 * logvar)
    var = std**2
    # 采样 epsilon
    eps = torch.randn(N, sample_K, D, device=mu.device)
    # 得到 z: [N, K, D]
    z = mu.unsqueeze(1) + eps * std.unsqueeze(1)
    # 必须开启梯度，因为后面要算 score
    z.requires_grad_(True)

    # === 2. 计算 score ===
    z_flat = z.view(N * sample_K, D)
    x_rep = x_target.unsqueeze(1)\
                     .expand(N, sample_K, *x_target.shape[1:])\
                     .reshape(N * sample_K, *x_target.shape[1:])
    
    # 调用修改后的 score_fn_torch，传入 prior_var
    scores_flat = score_fn_torch(z_flat, x_rep, decoder, sigma2=sigma2, prior_var=prior_var)
    scores = scores_flat.view(N, sample_K, D)

    # === 计算 Loss 项 ===
    # term1: 因为 prior_var = 1, 所以是 D / 1.0 = D
    
    
    # term2 和 term3 的计算保持原样，它们内部使用的 scores 已经包含了 sigma2 的影响
    g_norm2_sigma = (var.unsqueeze(1) * scores**2).sum(dim=2)
    align_term = ((z - mu.unsqueeze(1)) * scores).sum(dim=2)
    
    term1 = D / prior_var
    term2 = g_norm2_sigma.mean(dim=1)
    term3 = 2 * align_term.mean(dim=1) 

    # 如果你想做敏感性测试，记得加上 term1
    score_div =  term2 + term3
    
    loss = score_div.mean()  # 对 batch 求平均

    return loss

In [10]:
def cv_bamloss_batch(
    mu, logvar, x_target, decoder,
    sample_K=2, lambda_t=0.1, sigma2=0.1,
    writer=None, global_step=None,
):
    """
    向量化的 BaM 损失，支持 batch 并行 (集成 Control Variates 版本)。
    """
    N, D = mu.shape
    eps = 1e-8 # 防止除0

    # -------------------------------------------------------------------------
    # [Control Variates 准备阶段] 计算 g(mu) 和 grad(f1(mu))
    # -------------------------------------------------------------------------
    # 确保 mu 计算梯度
    mu.requires_grad_(True)
    
    # 1. 计算 g(mu) [N, D]
    # 注意：这里直接使用 x_target，因为 mu 对应每个 batch 只有 1 个
    g_mu = score_fn_torch(mu, x_target, decoder, sigma2)

    # 2. 计算 f1(mu) = g(mu)^T Sigma g(mu)
    std = torch.exp(0.5 * logvar)
    var = std**2
    f1_mu = (var * g_mu**2).sum(dim=1) # [N]

    # 3. 计算 grad(f1(mu)) [N, D]
    grad_f1_mu = torch.autograd.grad(
        outputs=f1_mu.sum(), inputs=mu, create_graph=True
    )[0]

    # -------------------------------------------------------------------------
    # [原有采样逻辑]
    # -------------------------------------------------------------------------
    # 1. 重参数化采样 z: [N, K, D]
    eps_z = torch.randn(N, sample_K, D, device=mu.device)
    z = mu.unsqueeze(1) + eps_z * std.unsqueeze(1)
    z.requires_grad_(True)

    # 2. 计算 score，先 reshape 到 [N*K, D]
    z_flat = z.view(N * sample_K, D)
    x_rep = x_target.unsqueeze(1)               \
                    .expand(N, sample_K, *x_target.shape[1:]) \
                    .reshape(N * sample_K, *x_target.shape[1:])
    scores_flat = score_fn_torch(z_flat, x_rep, decoder, sigma2)
    scores = scores_flat.view(N, sample_K, D)

    # -------------------------------------------------------------------------
    # [Control Variates 计算阶段]
    # -------------------------------------------------------------------------
    z_minus_mu = z - mu.unsqueeze(1) # [N, K, D]

    # 原始项 (Raw Terms) [N, K]
    # Term 2 raw: ||g_l||_Σ^2
    g_norm2_sigma = (var.unsqueeze(1) * scores**2).sum(dim=2) 
    # Term 3 raw (不含系数2): (z_l - μ)^T * g_l
    align_term = (z_minus_mu * scores).sum(dim=2)

    # 控制变量项 (CV Terms) [N, K]
    # CV1: grad(f1(mu))^T (z - mu)
    cv1_vals = (grad_f1_mu.unsqueeze(1) * z_minus_mu).sum(dim=2)
    # CV2: g(mu)^T (z - mu)
    cv2_vals = (g_mu.unsqueeze(1) * z_minus_mu).sum(dim=2)

    # 计算最优系数 c1, c2 (对 K 维度求统计量) [N]
    # c = Cov(Term, CV) / Var(CV)
    def compute_optimal_c(term_vals, cv_vals):
        # 1. 中心化
        term_centered = term_vals - term_vals.mean(dim=1, keepdim=True)
        cv_centered = cv_vals - cv_vals.mean(dim=1, keepdim=True)
        
        # 2. 分子：Covariance
        numerator = (term_centered * cv_centered).mean(dim=1)
        
        # 3. 分母：Variance
        # 【修改点1】加大 eps，从 1e-8 加到 1e-4 或 1e-5，避免 K=2 时方差过小导致的除零爆炸
        denominator = cv_vals.var(dim=1, unbiased=False) + 1e-5 
        
        c = numerator / denominator
        
        # 【修改点2】Detach！我们只希望 c 作为降低方差的系数，不希望优化器去通过改变 c 来优化 Loss
        c = c.detach()
        
        # 【修改点3】截断 (Clamping)！
        # 如果 c 计算出来是几千几万，说明 CV 和原始项的相关性出了问题或者方差估计完全错误。
        # 限制在 [-100, 100] 或更小的范围内，保证训练初期 Loss 不崩。
        #c = torch.clamp(c, min=-100.0, max=100.0)
        
        return c

    c1 = compute_optimal_c(g_norm2_sigma, cv1_vals)
    c2 = compute_optimal_c(align_term, cv2_vals)

    # -------------------------------------------------------------------------
    # [Score-matching 三项 (应用 CV)]
    # -------------------------------------------------------------------------
    
    # 1) tr(I) = D
    term1 = D
    
    # 2) Term 2: Mean(Raw - c1 * CV)
    # 1/L ∑ ||g_l||_Σ^2 - c1 * 1/L ∑ CV1
    term2_corrected = g_norm2_sigma - c1.unsqueeze(1) * cv1_vals
    term2 = term2_corrected.mean(dim=1) # [N]
    
    # 3) Term 3: Mean(Raw - c2 * CV) * 2
    # 2/L ∑ (z_l - mu)^T g_l - 2 * c2 * 1/L ∑ CV2
    term3_corrected = align_term - c2.unsqueeze(1) * cv2_vals
    term3 = 2 * term3_corrected.mean(dim=1) # [N]

    # 汇总 Loss
    score_div = term2 + term3

    # 6. 最终 loss 对 batch 求平均
    loss = score_div.mean() # 对 batch 求平均

    return loss

### eval

In [11]:
# # grayscale version (for MNIST)
# def eval_fid(model, test_loader, device, fid_metric):
#     model.eval()
#     fid_metric.reset()
#
#     with torch.no_grad():
#         for real, _ in test_loader:
#             real = real.to(device)
#
#             # 编码—采样—解码
#             mu, logvar = model.encode(real)
#             z = model.reparameterize(mu, logvar)
#             fake = model.decode(z)
#
#             # 从 [B, D] 恢复成 [B,1,side,side]
#             B, D = real.shape
#             side = int(D**0.5)
#             real_img = real.view(B, 1, side, side).repeat(1, 3, 1, 1)
#             fake_img = fake.view(B, 1, side, side).repeat(1, 3, 1, 1)
#
#             # resize 到 299×299
#             real_rs = F.interpolate(
#                 real_img, size=(299, 299), mode="bilinear", align_corners=False
#             )
#             fake_rs = F.interpolate(
#                 fake_img, size=(299, 299), mode="bilinear", align_corners=False
#             )
#
#             # 更新 FID 统计（uint8）
#             fid_metric.update((real_rs * 255).to(torch.uint8), real=True)
#             fid_metric.update((fake_rs * 255).to(torch.uint8), real=False)
#
#     # 计算并返回得分
#     fid_score = fid_metric.compute().item()
#     model.train()
#     return fid_score

In [12]:
def eval_fid(model, test_loader, device, fid_metric):
    model.eval()
    fid_metric.reset()

    with torch.no_grad():
        for real, _ in test_loader:
            real = real.to(device)

            # 编码—采样—解码
            mu, logvar = model.encode(real)
            z = model.reparameterize(mu, logvar)
            fake = model.decode(z)

            # 从 [B, D] 恢复成 [B,3,s,s]
            B, D = real.shape
            real_img = real.view(B, 3, s, s)
            fake_img = fake.view(B, 3, s, s)

            # resize 到 299×299
            real_rs = nn.functional.interpolate(real_img, size=(299, 299), mode="bilinear", align_corners=False)
            fake_rs = nn.functional.interpolate(fake_img, size=(299, 299), mode="bilinear", align_corners=False)

            # 更新 FID 统计（uint8）
            fid_metric.update((real_rs * 255).to(torch.uint8), real=True)
            fid_metric.update((fake_rs * 255).to(torch.uint8), real=False)

    # 计算并返回得分
    fid_score = fid_metric.compute().item()
    model.train()
    return fid_score

In [13]:
import math
@torch.no_grad()
def evaluate_iwae_gauss(model, dataloader, device, k=5000, sigma2=1.0):
    total_loglik = 0.0
    total_samples = 0
    D = None

    for x, _ in dataloader:
        x = x.to(device)
        B, D = x.shape
        total_samples += B

        # 编码
        mu, logvar = model.encode(x)  # [B, Z]
        std = torch.exp(0.5 * logvar)  # [B, Z]

        # 采样 k 个 z
        eps = torch.randn(B, k, mu.size(1), device=device)
        z = mu.unsqueeze(1) + std.unsqueeze(1) * eps  # [B, k, Z]

        # 第二项：log p(z) 标准正态先验
        log_pz = -0.5 * (z**2 + math.log(2 * math.pi)).sum(-1)  # [B, k]

        # 第三项：log q(z|x)
        log_qz = -0.5 * (
            ((z - mu.unsqueeze(1)) ** 2) / (std.unsqueeze(1) ** 2)
            + logvar.unsqueeze(1)
            + math.log(2 * math.pi)
        ).sum(
            -1
        )  # [B, k]

        # 解码
        x_recon = model.decode(z.view(B * k, -1))  # [B*k, D]
        x_expand = x.unsqueeze(1).repeat(1, k, 1).view(B * k, -1)
        diff = x_expand - x_recon  # [B*k, D]

        # 第一项：log p(x|z) 似然，固定方差 sigma2
        log_px = (
            -0.5 * (diff**2 / sigma2).sum(-1) - 0.5 * D * math.log(2 * math.pi * sigma2)
        ).view(
            B, k
        )  # [B, k]

        # importance weights: log-sum-exp 聚合
        log_w = log_px + log_pz - log_qz  # [B, k]
        m = log_w.max(dim=1, keepdim=True).values  # [B, 1]
        Lk = (
            m.squeeze(1) + torch.log(torch.exp(log_w - m).sum(dim=1)) - math.log(k)
        )  # [B]

        total_loglik += Lk.sum().item()

    avg_Lk = total_loglik / total_samples
    avg_NLL = -avg_Lk
    bpd = avg_NLL / (D * math.log(2))  # bits/dim
    return avg_Lk, avg_NLL, bpd

### plt

In [14]:
# # grayscale version (for MNIST)
# def write_test_images(writer, model, data_loader, device, global_step, sample_K=7,folder_name="img"):
#     model.eval()
#     batch, _ = next(iter(data_loader))
#     batch = batch.to(device)
#     K = min(sample_K, batch.size(0))
#     inputs = batch[:K].cpu().view(K, 28, 28)
#
#     with torch.no_grad():
#         mu, logvar = model.encode(batch)
#         sigma = torch.exp(0.5 * logvar)
#         eps = torch.randn_like(sigma)
#         z = mu + eps * sigma
#         recon = model.decode(z[:K]).cpu().view(K, 28, 28).numpy()
#
#     fig = plt.figure(figsize=(15, 6))
#     for i in range(K):
#         ax = fig.add_subplot(2, K, i + 1)
#         ax.imshow(inputs[i], cmap="gray")
#         ax.axis("off")
#         ax = fig.add_subplot(2, K, K + i + 1)
#         ax.imshow(recon[i], cmap="gray")
#         ax.axis("off")
#
#     writer.add_figure(f"{folder_name}/{global_step}-rec", fig, global_step)
#     plt.close(fig)

In [15]:
# RGB
def write_test_images(
    writer,
    model,
    data_loader,
    device,
    global_step,
    sample_K=7,
    img_size=(s, s),
    num_channels=3,
    folder_name="img",
):
    """
    在 TensorBoard 上展示 CIFAR-10 原图 & VAE 重构图。
    """
    model.eval()

    # 1) 从 data_loader 累积扁平张量 flat_imgs 直到 >= sample_K
    all_flat = []
    for flat_img, _ in data_loader:
        all_flat.append(flat_img)
        if sum(x.size(0) for x in all_flat) >= sample_K:
            break
    flat_imgs = torch.cat(all_flat, dim=0).to(device)

    B = flat_imgs.size(0)
    K = min(sample_K, B)
    C, H, W = num_channels, *img_size

    # 2) 把前 K 张先恢复成 [K, C, H, W] 供可视化
    inputs = flat_imgs[:K].view(K, C, H, W).cpu()

    # 3) 编码 → 样本 → 解码
    with torch.no_grad():
        mu, logvar = model.encode(flat_imgs)
        sigma = torch.exp(0.5 * logvar)
        eps = torch.randn_like(sigma)
        z = mu + eps * sigma

        recon_flat = model.decode(z[:K]).cpu()
        recon = recon_flat.view(K, C, H, W)

    # 4) 转成 HWC NumPy 数组
    inputs_np = inputs.permute(0, 2, 3, 1).contiguous().numpy()
    recon_np = recon.permute(0, 2, 3, 1).contiguous().numpy()

    # 5) 按组绘图，每组最多 7 张
    group_size = 7
    for group_idx in range(0, K, group_size):
        end_idx = min(group_idx + group_size, K)
        current_K = end_idx - group_idx

        fig = plt.figure(figsize=(2 * current_K, 4))
        for i in range(current_K):
            ax1 = fig.add_subplot(2, current_K, i + 1)
            ax1.imshow(inputs_np[group_idx + i])
            ax1.axis("off")

            ax2 = fig.add_subplot(2, current_K, current_K + i + 1)
            ax2.imshow(recon_np[group_idx + i])
            ax2.axis("off")

        writer.add_figure(
            f"{folder_name}/{global_step}-{group_idx // group_size}",
            fig,
            global_step,
        )
        plt.close(fig)
    model.train()

In [16]:
def log_latent_space(writer, model, dataloader, global_step, device):
    model.eval()
    all_mu = []
    all_logvar = []
    all_labels = []
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            mu, logvar = model.encode(x)
            all_mu.append(mu.cpu())
            all_logvar.append(logvar.cpu())
            all_labels.extend(y.cpu().tolist())

    all_mu = torch.cat(all_mu, dim=0)  # [N, z_dim]
    all_logvar = torch.cat(all_logvar, dim=0)  # [N, z_dim]
    all_labels = [str(l) for l in all_labels]  # 转成字符串，方便 Color by

    max_points = 2000
    all_mu = all_mu[:max_points]
    all_labels = all_labels[:max_points]
    all_logvar = all_logvar[:max_points]

    # print("mu 是否有nan: ", torch.isnan(all_mu).any(), torch.isinf(all_mu).any())
    # print("logvar 是否有nan: ", torch.isnan(all_logvar).any(), torch.isinf(all_logvar).any())
    # print("数据维度: ", all_mu.shape, all_logvar.shape, len(all_labels))
    # print("数据范围: ", all_mu.min().item(), all_mu.max().item(), all_logvar.min().item(), all_logvar.max().item())
    # print("数据方差: ", all_mu.std(dim=0), all_logvar.std(dim=0))

    # print("标签类别: ", set(all_labels))
    # 写入 TensorBoard Embedding
    writer.add_embedding(
        mat=all_mu,
        metadata=all_labels,  # 标签（比如数字类别）
        tag="latent_mu",
        global_step=global_step,
    )
    writer.add_embedding(
        mat=all_logvar,
        metadata=all_labels,  # 标签（比如数字类别）
        tag="latent_logvar",
        global_step=global_step,
    )

In [17]:
def log_latent_stats(model, loader, writer, epoch, device):
    model.eval()
    ############## 写 mu/std/z m##############
    all_mu, all_std, all_z = [], [], []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            mu, logvar = model.encode(x)  # [B, Z]
            std = torch.exp(0.5 * logvar)  # [B, Z]
            # 如果你想按重参数化采样的 z
            eps = torch.randn_like(std)
            z = mu + eps * std  # [B, Z]

            all_mu.append(mu.cpu())
            all_std.append(std.cpu())
            all_z.append(z.cpu())

    # 拼接成 [N, Z]
    all_mu = torch.cat(all_mu, dim=0)
    all_std = torch.cat(all_std, dim=0)
    all_z = torch.cat(all_z, dim=0)

    # 对样本维度取均值 → [Z]
    mu_mean = all_mu.mean(dim=0)  # Tensor of shape [Z]
    std_mean = all_std.mean(dim=0)
    z_mean = all_z.mean(dim=0)

    # 控制台打印（可选）
    print(f"Epoch {epoch+1}  mu_mean per dim:  {mu_mean.tolist()}")
    print(f"Epoch {epoch+1}  std_mean per dim: {std_mean.tolist()}")
    print(f"Epoch {epoch+1}  z_mean per dim:   {z_mean.tolist()}")

    # 写入 TensorBoard：每个潜变量维度一个 scalar
    for i in range(mu_mean.size(0)):
        writer.add_scalar(f"latent/mu_dim{i}", mu_mean[i].item(), epoch)
        writer.add_scalar(f"latent/std_dim{i}", std_mean[i].item(), epoch)
        writer.add_scalar(f"latent/z_dim{i}", z_mean[i].item(), epoch)

    # 如果还想看分布，可写 histogram：
    writer.add_histogram("histogram/mu", all_mu, epoch)
    writer.add_histogram("histogram/std", all_std, epoch)
    writer.add_histogram("histogram/z", all_z, epoch)

    ############## 写 g_dim ##############
    Z = all_mu.size(1)  # 潜变量维度 5

    for d in range(Z):
        vals = epoch_metrics_g[f"g_abs_dim{d}"]  # list of length num_batches
        avg = sum(vals) / len(vals)  # 维度 d 上的平均梯度
        writer.add_scalar(f"latent_g/g_abs_dim{d}", avg, epoch)

        vals = epoch_metrics_g[f"g_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g/g_dim{d}", avg, epoch)

    # 写 g 的整体分布直方图
    all_g = []
    for d in range(Z):
        all_g.extend(epoch_metrics_g[f"g_dim{d}"])  # 收集所有维度的 g

    all_g_tensor = torch.tensor(all_g, dtype=torch.float32)
    writer.add_histogram("histogram_g/g", all_g_tensor, epoch)

    print(f"Epoch {epoch+1}  g per dim: {[epoch_metrics_g[f'g_abs_dim{d}'][-1] for d in range(Z)]}")

    ############### 写 g 的分解项 ##############
    for d in range(Z):
        vals = epoch_metrics_g[f"g_pz_abs_dim{d}"]  # list of length num_batches
        avg = sum(vals) / len(vals)  # 维度 d 上的平均
        writer.add_scalar(f"latent_g_pz/g_pz_abs_dim{d}", avg, epoch)
        vals = epoch_metrics_g[f"g_px_abs_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_px/g_px_abs_dim{d}", avg, epoch)

        vals = epoch_metrics_g[f"g_pz_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_pz/g_pz_dim{d}", avg, epoch)
        vals = epoch_metrics_g[f"g_px_dim{d}"]
        avg = sum(vals) / len(vals)
        writer.add_scalar(f"latent_g_px/g_px_dim{d}", avg, epoch)

    # g_pz 和 g_px 的整体分布直方图
    all_g_pz = []
    all_g_px = []
    for d in range(Z):            
        all_g_pz.extend(epoch_metrics_g[f"g_pz_dim{d}"])
        all_g_px.extend(epoch_metrics_g[f"g_px_dim{d}"])

    all_g_pz_tensor = torch.tensor(all_g_pz, dtype=torch.float32)
    all_g_px_tensor = torch.tensor(all_g_px, dtype=torch.float32)
    writer.add_histogram("histogram_g/g_pz", all_g_pz_tensor, epoch)
    writer.add_histogram("histogram_g/g_px", all_g_px_tensor, epoch)

    print(f"Epoch {epoch+1}  g_pz per dim: {[epoch_metrics_g[f'g_pz_abs_dim{d}'][-1] for d in range(Z)]}")
    print(f"Epoch {epoch+1}  g_px per dim: {[epoch_metrics_g[f'g_px_abs_dim{d}'][-1] for d in range(Z)]}")

    model.train()  # 别忘了恢复训练模式

## run cif100

In [18]:
#### ours
batch_size0 = 128
batch_size_test = 64
batch_size_test_nll = 16
epoch0 = 100
lr0 = 1e-4


sigma20 = 1
prior_var0 = 1
lambda_t0 = 0.1
lambda_KLt = 0.1
lambda_D0 = 0.5
lambda_KL0 = 0
lambda_sum = 0.5 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1}
lambda_KL0 = 0 # vae的 KL，非vae时=0

torch.cuda.empty_cache()

device = torch.device(f"cuda:{cuda_num}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

flag_vae = False
flag_iwae = False
flag_cv = False
prefix = f"F3-add-latent{LATENT_DIM}"
folder = "runs__re"
file_idx = 4444
sample_K0 = 2

lambda_D0 = 0.5
lambda_sum = 1 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1} 的分母
num_KLt = 1.0 # BaM 的 KL{t+1} 的分子

sigma20 = 1
prior_var0 = 1
sigma2_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
prior_var_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
sigma2_list = [0.01, 1.0, 10.0]
prior_var_list = [0.01, 1.0, 10.0]
for lr0 in [5e-4]:
    for sigma20 in [1]:
        for prior_var0 in prior_var_list:    
            for tensround in range(file_idx, file_idx + 1):
                time_now = datetime.now().strftime("%m-%d_%H-%M")
                if flag_vae:
                    lambda_sum = 0
                    lambda_KLt = 1
                    lambda_KL0 = 1
                    model_dir=f"{folder}/{prefix}-{datasetname}-vae-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir 
                    )
                elif flag_iwae:
                    model_dir=f"{folder}/{prefix}-{datasetname}-iwae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                elif flag_cv:
                    model_dir=f"{folder}/{prefix}-{datasetname}-cv-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                else:
                    model_dir=f"{folder}/{prefix}-{datasetname}-ours-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{num_KLt/lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                    log_dir=model_dir
                )
                model = VAE().to(device)
                model_t = VAE().to(device)
                optimizer = torch.optim.Adam(model.parameters(), lr=lr0)

                fid_metric = FrechetInceptionDistance(feature=2048).to(device)

                data_loader = DataLoader(
                    train_dataset_torch, batch_size=batch_size0, shuffle=True,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                val_loader = DataLoader(
                    val_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4, worker_init_fn=seed_worker, generator=gen
                )
                                

                test_loader = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                test_loader_nll = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test_nll, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )

                plt_ck_p = epoch0 // 20  # 显示 20 次测试结果

                total_steps = epoch0 * len(data_loader)

                for epoch in range(epoch0):

                    losses = []
                    Dqs = []
                    KLs = []
                    KL_ts = []
                    MSEs = []
                    BCEs = []
                    sum_Dq_plus_lKLt_dot_KLt_s = []

                    alpha_max = 1.0
                    beta_max = 1.0
                    gamma_max = 1.0

                    warmup1 = 30      # 只用 MSE+KL
                    warmup2 = 80      # 完整过渡到 MSE+KL+BaM
                    # alpha = alpha_max * min(1.0, epoch / warmup1)
                    alpha = alpha_max
                    gamma = gamma_max
                    beta = beta_max
                    
                    old_enc_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.encoder.state_dict().items()
                    }

                    for batch_idx, (x, y) in enumerate(data_loader):
                        x = x.to(device)  # [batch, ...]
                        recon_batch, mu, logvar = model(x)  # [batch, D]

                        # 恢复旧 encoder 的参数到临时 old_encoder 推断（在 CPU 拷贝上不需要梯度）
                        with torch.no_grad():
                            # 将保存的 old state 恢复到 model.encoder 的一个临时副本计算 mu_old/logvar_old
                            # 更高效：直接在当前 encoder 上 load_state_dict(old_enc_state)，但要小心线程安全与 GPU；这里用简单方式：
                            # 创建新 encoder 实例会更健壮但也更重 —— 下面采用直接把旧参数载入 model.encoder 并再恢复当前参数的办法（较快）
                            model_t.encoder.load_state_dict(old_enc_state)   # load old weights
                            # 注意：mu_old, logvar_old 都是在 no_grad 环境下计算的（无需保持 grad）
                            mu_old, logvar_old = model_t.encoder(x)         # compute old outputs

                        global_step = epoch * len(data_loader) + batch_idx

                        KL = 1.0 / x.size(0) * loss_function(x, recon_batch, mu, logvar, sigma2=sigma20, prior_var=prior_var0)
                        KLs.append(KL.item())
                        
                        if not flag_vae and not flag_iwae:                            
                            if flag_cv:
                                Dq = cv_bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K1=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                            else:
                                Dq = bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    prior_var=prior_var0)
                        else:
                            Dq = torch.tensor(0.0)
                        Dqs.append(Dq.item())
                        
                        MSE = nn.functional.mse_loss(
                            recon_batch, x, reduction="sum"
                        ) / x.size(0)

                        MSEs.append(MSE.item())
                        MSE = (1.0 / sigma20) * MSE

                        var = torch.exp(logvar)
                        var_old = torch.exp(logvar_old)
                        # 先在 latent 维度 D 上求和
                        KL_t_per_latent = 0.5 * (
                            torch.log(var / var_old) +
                            (var_old + (mu_old - mu).pow(2)) / var - 1
                        ).sum(dim=1)                        # [N]
                        # 再对 batch 维度求均值
                        KL_t = KL_t_per_latent.mean()        # scalar

                        KL_ts.append(KL_t.item())

                        Dq *= lambda_D0
                        KL *= lambda_KL0
                        KL_t *= num_KLt / lambda_KLt

                        sum_Dq_plus_lKLt_dot_KLt = Dq + KL_t
                        sum_Dq_plus_lKLt_dot_KLt_s.append(sum_Dq_plus_lKLt_dot_KLt.item())
                        sum_Dq_plus_lKLt_dot_KLt *= lambda_sum
                        
                        if flag_iwae:
                            loss = 1 / x.size(0) * loss_iwae_gauss(mu, logvar, x, model.decode, k=sample_K0)
                        elif flag_vae:
                            loss = MSE + KL
                        else:
                            loss = MSE + sum_Dq_plus_lKLt_dot_KLt


                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                        losses.append(loss.item())

                    # 3. 记录每个 epoch 的平均 loss
                    mean_loss = np.mean(losses)
                    mean_MSE = np.mean(MSEs)
                    # mean_BCE = np.mean(BCEs)
                    mean_Dq = np.mean(Dqs)
                    mean_KL = np.mean(KLs)
                    mean_KL_t = np.mean(KL_ts)
                    mean_Dq_plus_lKLt_dot_KLt = np.mean(sum_Dq_plus_lKLt_dot_KLt_s)

                    writer.add_scalar("Loss/train_epoch", mean_loss, epoch)
                    writer.add_scalar("MSE/train_epoch", mean_MSE, epoch)
                    # writer.add_scalar("BCE/train_epoch", mean_BCE, epoch)
                    writer.add_scalar("Dq/train_epoch", mean_Dq, epoch)
                    writer.add_scalar("KL/train_epoch", mean_KL, epoch)
                    writer.add_scalar("KL_t/train_epoch", mean_KL_t, epoch)
                    writer.add_scalar("Sum/train_epoch", mean_Dq_plus_lKLt_dot_KLt, epoch)

                    write_test_images(writer, model, test_loader, device, epoch)
                    
                    fid_score = eval_fid(model, val_loader, device, fid_metric)
                    writer.add_scalar("FID/val_epoch", fid_score, epoch)
                    print(
                        f"[Epoch {epoch+1}] Loss: {mean_loss:.4f}, FID: {fid_score:.4f}, BCE: {'N/A'}"
                    )
                    print(
                        f"           MSE: {mean_MSE:.4f}, Dq: {mean_Dq:.4f}, KL: {mean_KL:.4f}, KL_t: {mean_KL_t:.4f}"
                    )
                    torch.cuda.empty_cache()
                    
                write_test_images(
                    writer,
                    model,
                    test_loader,
                    device,
                    global_step=0,
                    sample_K=42,
                    folder_name="img_final_epoch",
                )
                
                fid_score_final = eval_fid(model, test_loader, device, fid_metric)
                writer.add_scalar("FID/test_epoch", fid_score_final, epoch)
                print(f"Final Test FID: {fid_score_final:.4f}")

                hparam_dict = {
                    "batch_size": batch_size0,
                    "epochs": epoch0,
                    "lr": lr0,
                    "K": sample_K0,
                    "sigma2": sigma20,
                    "lambda_D": lambda_D0,
                    "lambda_KL": lambda_KL0,
                    "lambda_KLt": lambda_KLt,
                    "lambda_sum": lambda_sum,
                    "tensround": tensround,
                }
                metric_dict = {
                    "last loss": mean_loss,
                    "last FID": float(fid_score_final),
                    "last MSE": mean_MSE,
                }
                writer.add_hparams(hparam_dict, metric_dict)
                # 4. 训练结束后关闭 writer
                writer.close()
                
                torch.save(
                    model,
                    f"{model_dir}/final_model.pt",
                )
                

Using device: cuda:2


/home/yzm/anaconda3/envs/gen/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:36: UserWarning: Metric `FrechetInceptionDistance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


[Epoch 1] Loss: 1022.4290, FID: 412.0934, BCE: N/A
           MSE: 217.5662, Dq: 1598.6679, KL: 29.8177, KL_t: 276.4420
[Epoch 2] Loss: 210.9234, FID: 416.1382, BCE: N/A
           MSE: 210.3948, Dq: 1.0349, KL: 3.3301, KL_t: 0.5574
[Epoch 3] Loss: 161.6263, FID: 408.2458, BCE: N/A
           MSE: 162.4840, Dq: -1.7994, KL: 7.8714, KL_t: 2.1002
[Epoch 4] Loss: 132.4914, FID: 382.4109, BCE: N/A
           MSE: 136.2809, Dq: -7.7303, KL: 10.9518, KL_t: 3.7788
[Epoch 5] Loss: 110.2857, FID: 361.2007, BCE: N/A
           MSE: 114.2816, Dq: -8.1001, KL: 14.7361, KL_t: 2.7071
[Epoch 6] Loss: 98.6491, FID: 362.2732, BCE: N/A
           MSE: 103.1617, Dq: -9.1388, KL: 16.8608, KL_t: 2.8402
[Epoch 7] Loss: 90.8900, FID: 353.3211, BCE: N/A
           MSE: 95.9211, Dq: -10.1908, KL: 18.5043, KL_t: 3.2151
[Epoch 8] Loss: 88.4030, FID: 343.2583, BCE: N/A
           MSE: 93.8454, Dq: -11.0866, KL: 19.0089, KL_t: 5.0470
[Epoch 9] Loss: 85.3075, FID: 343.4440, BCE: N/A
           MSE: 90.7366, Dq: -11

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-nreof8c3'


[Epoch 31] Loss: 56.9046, FID: 226.1050, BCE: N/A
           MSE: 59.6436, Dq: -6.1313, KL: 40.6224, KL_t: 16.3319
[Epoch 32] Loss: 56.3357, FID: 224.4496, BCE: N/A
           MSE: 59.1446, Dq: -6.1859, KL: 41.0636, KL_t: 14.2023
[Epoch 33] Loss: 55.9558, FID: 221.1257, BCE: N/A
           MSE: 58.7993, Dq: -6.2485, KL: 40.9885, KL_t: 14.0404
[Epoch 34] Loss: 56.1973, FID: 226.2883, BCE: N/A
           MSE: 58.9772, Dq: -6.1510, KL: 41.4230, KL_t: 14.7808
[Epoch 35] Loss: 55.6599, FID: 224.0499, BCE: N/A
           MSE: 58.5023, Dq: -6.2169, KL: 41.9528, KL_t: 13.3056
[Epoch 36] Loss: 55.9646, FID: 228.6044, BCE: N/A
           MSE: 58.6984, Dq: -6.2283, KL: 40.6953, KL_t: 19.0144
[Epoch 37] Loss: 55.8015, FID: 221.9583, BCE: N/A
           MSE: 58.5256, Dq: -6.0592, KL: 41.5728, KL_t: 15.2769
[Epoch 38] Loss: 55.3206, FID: 218.3743, BCE: N/A
           MSE: 57.9077, Dq: -5.7962, KL: 41.6177, KL_t: 15.5516
[Epoch 39] Loss: 55.3213, FID: 220.1886, BCE: N/A
           MSE: 57.8176, Dq: -

...

[Epoch 94] Loss: 47.8486, FID: 193.7752, BCE: N/A
           MSE: 49.3147, Dq: -3.1767, KL: 50.2052, KL_t: 6.1123


Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-b_8w5ppl'


[Epoch 95] Loss: 48.0697, FID: 192.8517, BCE: N/A
           MSE: 49.5295, Dq: -3.2033, KL: 49.9211, KL_t: 7.0908
[Epoch 96] Loss: 48.0625, FID: 196.3059, BCE: N/A
           MSE: 49.5083, Dq: -3.2121, KL: 49.4064, KL_t: 8.0095
[Epoch 97] Loss: 47.8548, FID: 192.1443, BCE: N/A
           MSE: 49.3116, Dq: -3.1679, KL: 49.6596, KL_t: 6.3582
[Epoch 98] Loss: 47.9320, FID: 195.7484, BCE: N/A
           MSE: 49.4016, Dq: -3.2259, KL: 49.2075, KL_t: 7.1678
[Epoch 99] Loss: 48.0669, FID: 189.8603, BCE: N/A
           MSE: 49.5102, Dq: -3.1819, KL: 49.2078, KL_t: 7.3851
[Epoch 100] Loss: 47.9850, FID: 192.2667, BCE: N/A
           MSE: 49.4174, Dq: -3.1466, KL: 49.4140, KL_t: 7.0436
Final Test FID: 188.4222
[Epoch 1] Loss: 118.3139, FID: 361.1173, BCE: N/A
           MSE: 114.4996, Dq: -0.2950, KL: 38.8232, KL_t: 198.0902
[Epoch 2] Loss: 73.5783, FID: 340.4013, BCE: N/A
           MSE: 74.3518, Dq: -2.6752, KL: 52.9136, KL_t: 28.2050
[Epoch 3] Loss: 63.1041, FID: 304.8362, BCE: N/A
          

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-qk0_utzn'


[Epoch 31] Loss: 50.1677, FID: 212.0829, BCE: N/A
           MSE: 50.9664, Dq: -1.9554, KL: 71.4749, KL_t: 8.9481
[Epoch 32] Loss: 50.1052, FID: 210.5946, BCE: N/A
           MSE: 50.8977, Dq: -1.9372, KL: 71.5654, KL_t: 8.8036
[Epoch 33] Loss: 49.9289, FID: 209.5913, BCE: N/A
           MSE: 50.7000, Dq: -1.8787, KL: 71.9252, KL_t: 8.4086
[Epoch 34] Loss: 49.8922, FID: 211.0119, BCE: N/A
           MSE: 50.6654, Dq: -1.8905, KL: 71.9463, KL_t: 8.5988
[Epoch 35] Loss: 50.0196, FID: 208.2410, BCE: N/A
           MSE: 50.7605, Dq: -1.8583, KL: 71.9865, KL_t: 9.4113
[Epoch 36] Loss: 49.7000, FID: 206.8797, BCE: N/A
           MSE: 50.4943, Dq: -1.9076, KL: 71.8636, KL_t: 7.9721
[Epoch 37] Loss: 49.8157, FID: 209.6942, BCE: N/A
           MSE: 50.6045, Dq: -1.9251, KL: 71.8396, KL_t: 8.6852
[Epoch 38] Loss: 49.7864, FID: 207.8075, BCE: N/A
           MSE: 50.5267, Dq: -1.8244, KL: 72.1612, KL_t: 8.5919
[Epoch 39] Loss: 49.5753, FID: 206.2287, BCE: N/A
           MSE: 50.3580, Dq: -1.8916, 

In [19]:
#### ours
batch_size0 = 128
batch_size_test = 64
batch_size_test_nll = 16
epoch0 = 100
lr0 = 1e-4


sigma20 = 1
prior_var0 = 1
lambda_t0 = 0.1
lambda_KLt = 0.1
lambda_D0 = 0.5
lambda_KL0 = 0
lambda_sum = 0.5 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1}
lambda_KL0 = 0 # vae的 KL，非vae时=0

torch.cuda.empty_cache()

device = torch.device(f"cuda:{cuda_num}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

flag_vae = True
flag_iwae = False
flag_cv = False
prefix = f"F3-add-latent{LATENT_DIM}"
folder = "runs__re"
file_idx = 4444
sample_K0 = 2

lambda_D0 = 0.5
lambda_sum = 1 # BaM 的总系数
lambda_KLt = 50 # BaM 的 KL{t+1} 的分母
num_KLt = 1.0 # BaM 的 KL{t+1} 的分子

sigma20 = 1
prior_var0 = 1
sigma2_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
prior_var_list = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
sigma2_list = [0.01, 1.0, 10.0]
prior_var_list = [0.01, 1.0, 10.0]
for lr0 in [5e-4]:
    for sigma20 in [1]:
        for prior_var0 in prior_var_list:    
            for tensround in range(file_idx, file_idx + 1):
                time_now = datetime.now().strftime("%m-%d_%H-%M")
                if flag_vae:
                    lambda_sum = 0
                    lambda_KLt = 1
                    lambda_KL0 = 1
                    model_dir=f"{folder}/{prefix}-{datasetname}-vae-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir 
                    )
                elif flag_iwae:
                    model_dir=f"{folder}/{prefix}-{datasetname}-iwae-lr{lr0}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                elif flag_cv:
                    model_dir=f"{folder}/{prefix}-{datasetname}-cv-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                        log_dir=model_dir
                    )
                else:
                    model_dir=f"{folder}/{prefix}-{datasetname}-ours-s2lk{sigma20}-s2p{prior_var0}-lr{lr0}-K{sample_K0}-ls{lambda_sum}-lKLt{num_KLt/lambda_KLt}-{tensround}-{time_now}"
                    writer = SummaryWriter(
                    log_dir=model_dir
                )
                model = VAE().to(device)
                model_t = VAE().to(device)
                optimizer = torch.optim.Adam(model.parameters(), lr=lr0)

                fid_metric = FrechetInceptionDistance(feature=2048).to(device)

                data_loader = DataLoader(
                    train_dataset_torch, batch_size=batch_size0, shuffle=True,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                val_loader = DataLoader(
                    val_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4, worker_init_fn=seed_worker, generator=gen
                )
                                

                test_loader = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )
                
                test_loader_nll = DataLoader(
                    test_dataset_torch, batch_size=batch_size_test_nll, shuffle=False,
                    num_workers=4,                 # 如果 >0，则需要 seed_worker
                    worker_init_fn=seed_worker,    # 每个 worker 的随机种子
                    generator=gen                    # shuffle 用的随机数生成器
                )

                plt_ck_p = epoch0 // 20  # 显示 20 次测试结果

                total_steps = epoch0 * len(data_loader)

                for epoch in range(epoch0):

                    losses = []
                    Dqs = []
                    KLs = []
                    KL_ts = []
                    MSEs = []
                    BCEs = []
                    sum_Dq_plus_lKLt_dot_KLt_s = []

                    alpha_max = 1.0
                    beta_max = 1.0
                    gamma_max = 1.0

                    warmup1 = 30      # 只用 MSE+KL
                    warmup2 = 80      # 完整过渡到 MSE+KL+BaM
                    # alpha = alpha_max * min(1.0, epoch / warmup1)
                    alpha = alpha_max
                    gamma = gamma_max
                    beta = beta_max
                    
                    old_enc_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.encoder.state_dict().items()
                    }

                    for batch_idx, (x, y) in enumerate(data_loader):
                        x = x.to(device)  # [batch, ...]
                        recon_batch, mu, logvar = model(x)  # [batch, D]

                        # 恢复旧 encoder 的参数到临时 old_encoder 推断（在 CPU 拷贝上不需要梯度）
                        with torch.no_grad():
                            # 将保存的 old state 恢复到 model.encoder 的一个临时副本计算 mu_old/logvar_old
                            # 更高效：直接在当前 encoder 上 load_state_dict(old_enc_state)，但要小心线程安全与 GPU；这里用简单方式：
                            # 创建新 encoder 实例会更健壮但也更重 —— 下面采用直接把旧参数载入 model.encoder 并再恢复当前参数的办法（较快）
                            model_t.encoder.load_state_dict(old_enc_state)   # load old weights
                            # 注意：mu_old, logvar_old 都是在 no_grad 环境下计算的（无需保持 grad）
                            mu_old, logvar_old = model_t.encoder(x)         # compute old outputs

                        global_step = epoch * len(data_loader) + batch_idx

                        KL = 1.0 / x.size(0) * loss_function(x, recon_batch, mu, logvar, sigma2=sigma20, prior_var=prior_var0)
                        KLs.append(KL.item())
                        
                        if not flag_vae and not flag_iwae:                            
                            if flag_cv:
                                Dq = cv_bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K1=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    writer=writer, global_step=global_step)
                            else:
                                Dq = bamloss_batch(mu, logvar, x, model.decoder,
                                    sample_K=sample_K0, lambda_t=lambda_t0, sigma2=sigma20,
                                    prior_var=prior_var0)
                        else:
                            Dq = torch.tensor(0.0)
                        Dqs.append(Dq.item())
                        
                        MSE = nn.functional.mse_loss(
                            recon_batch, x, reduction="sum"
                        ) / x.size(0)

                        MSEs.append(MSE.item())
                        MSE = (1.0 / sigma20) * MSE

                        var = torch.exp(logvar)
                        var_old = torch.exp(logvar_old)
                        # 先在 latent 维度 D 上求和
                        KL_t_per_latent = 0.5 * (
                            torch.log(var / var_old) +
                            (var_old + (mu_old - mu).pow(2)) / var - 1
                        ).sum(dim=1)                        # [N]
                        # 再对 batch 维度求均值
                        KL_t = KL_t_per_latent.mean()        # scalar

                        KL_ts.append(KL_t.item())

                        Dq *= lambda_D0
                        KL *= lambda_KL0
                        KL_t *= num_KLt / lambda_KLt

                        sum_Dq_plus_lKLt_dot_KLt = Dq + KL_t
                        sum_Dq_plus_lKLt_dot_KLt_s.append(sum_Dq_plus_lKLt_dot_KLt.item())
                        sum_Dq_plus_lKLt_dot_KLt *= lambda_sum
                        
                        if flag_iwae:
                            loss = 1 / x.size(0) * loss_iwae_gauss(mu, logvar, x, model.decode, k=sample_K0)
                        elif flag_vae:
                            loss = MSE + KL
                        else:
                            loss = MSE + sum_Dq_plus_lKLt_dot_KLt


                        optimizer.zero_grad()
                        loss.backward()
                        optimizer.step()
                        losses.append(loss.item())

                    # 3. 记录每个 epoch 的平均 loss
                    mean_loss = np.mean(losses)
                    mean_MSE = np.mean(MSEs)
                    # mean_BCE = np.mean(BCEs)
                    mean_Dq = np.mean(Dqs)
                    mean_KL = np.mean(KLs)
                    mean_KL_t = np.mean(KL_ts)
                    mean_Dq_plus_lKLt_dot_KLt = np.mean(sum_Dq_plus_lKLt_dot_KLt_s)

                    writer.add_scalar("Loss/train_epoch", mean_loss, epoch)
                    writer.add_scalar("MSE/train_epoch", mean_MSE, epoch)
                    # writer.add_scalar("BCE/train_epoch", mean_BCE, epoch)
                    writer.add_scalar("Dq/train_epoch", mean_Dq, epoch)
                    writer.add_scalar("KL/train_epoch", mean_KL, epoch)
                    writer.add_scalar("KL_t/train_epoch", mean_KL_t, epoch)
                    writer.add_scalar("Sum/train_epoch", mean_Dq_plus_lKLt_dot_KLt, epoch)

                    write_test_images(writer, model, test_loader, device, epoch)
                    
                    fid_score = eval_fid(model, val_loader, device, fid_metric)
                    writer.add_scalar("FID/val_epoch", fid_score, epoch)
                    print(
                        f"[Epoch {epoch+1}] Loss: {mean_loss:.4f}, FID: {fid_score:.4f}, BCE: {'N/A'}"
                    )
                    print(
                        f"           MSE: {mean_MSE:.4f}, Dq: {mean_Dq:.4f}, KL: {mean_KL:.4f}, KL_t: {mean_KL_t:.4f}"
                    )
                    torch.cuda.empty_cache()
                    
                write_test_images(
                    writer,
                    model,
                    test_loader,
                    device,
                    global_step=0,
                    sample_K=42,
                    folder_name="img_final_epoch",
                )
                
                fid_score_final = eval_fid(model, test_loader, device, fid_metric)
                writer.add_scalar("FID/test_epoch", fid_score_final, epoch)
                print(f"Final Test FID: {fid_score_final:.4f}")

                hparam_dict = {
                    "batch_size": batch_size0,
                    "epochs": epoch0,
                    "lr": lr0,
                    "K": sample_K0,
                    "sigma2": sigma20,
                    "lambda_D": lambda_D0,
                    "lambda_KL": lambda_KL0,
                    "lambda_KLt": lambda_KLt,
                    "lambda_sum": lambda_sum,
                    "tensround": tensround,
                }
                metric_dict = {
                    "last loss": mean_loss,
                    "last FID": float(fid_score_final),
                    "last MSE": mean_MSE,
                }
                writer.add_hparams(hparam_dict, metric_dict)
                # 4. 训练结束后关闭 writer
                writer.close()
                
                torch.save(
                    model,
                    f"{model_dir}/final_model.pt",
                )
                

Using device: cuda:2


Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-6xbx2haa'


[Epoch 1] Loss: 187.6591, FID: 371.3336, BCE: N/A
           MSE: 164.0502, Dq: 0.0000, KL: 23.6090, KL_t: 1057.9441
[Epoch 2] Loss: 123.2502, FID: 338.9184, BCE: N/A
           MSE: 110.1103, Dq: 0.0000, KL: 13.1399, KL_t: 3.1905
[Epoch 3] Loss: 112.8930, FID: 344.7108, BCE: N/A
           MSE: 98.6923, Dq: 0.0000, KL: 14.2007, KL_t: 3.1677
[Epoch 4] Loss: 105.5805, FID: 298.2158, BCE: N/A
           MSE: 90.3879, Dq: 0.0000, KL: 15.1926, KL_t: 5.9501
[Epoch 5] Loss: 102.0745, FID: 287.0317, BCE: N/A
           MSE: 86.6352, Dq: 0.0000, KL: 15.4392, KL_t: 3.2069
[Epoch 6] Loss: 98.2262, FID: 276.8542, BCE: N/A
           MSE: 82.4186, Dq: 0.0000, KL: 15.8076, KL_t: 3.6429
[Epoch 7] Loss: 95.8142, FID: 268.2717, BCE: N/A
           MSE: 79.5283, Dq: 0.0000, KL: 16.2859, KL_t: 4.0108
[Epoch 8] Loss: 93.1317, FID: 261.3838, BCE: N/A
           MSE: 76.6710, Dq: 0.0000, KL: 16.4607, KL_t: 2.9914
[Epoch 9] Loss: 91.2718, FID: 253.2470, BCE: N/A
           MSE: 74.6908, Dq: 0.0000, KL: 16.5

Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-e2y005a5'


[Epoch 12] Loss: 88.0554, FID: 236.4047, BCE: N/A
           MSE: 70.7643, Dq: 0.0000, KL: 17.2912, KL_t: 5.5622
[Epoch 13] Loss: 87.1983, FID: 231.2372, BCE: N/A
           MSE: 69.5668, Dq: 0.0000, KL: 17.6315, KL_t: 5.3213
[Epoch 14] Loss: 87.0600, FID: 233.1937, BCE: N/A
           MSE: 69.2202, Dq: 0.0000, KL: 17.8398, KL_t: 4.0319
[Epoch 15] Loss: 86.8920, FID: 231.5876, BCE: N/A
           MSE: 69.0047, Dq: 0.0000, KL: 17.8873, KL_t: 4.6329
[Epoch 16] Loss: 86.0079, FID: 230.3106, BCE: N/A
           MSE: 68.0193, Dq: 0.0000, KL: 17.9886, KL_t: 4.2530
[Epoch 17] Loss: 86.0541, FID: 229.6824, BCE: N/A
           MSE: 67.8859, Dq: 0.0000, KL: 18.1682, KL_t: 4.3101
[Epoch 18] Loss: 85.4080, FID: 226.5726, BCE: N/A
           MSE: 66.9608, Dq: 0.0000, KL: 18.4472, KL_t: 3.3856
[Epoch 19] Loss: 85.3163, FID: 223.4824, BCE: N/A
           MSE: 66.6389, Dq: 0.0000, KL: 18.6775, KL_t: 5.0681
[Epoch 20] Loss: 84.9878, FID: 228.8494, BCE: N/A
           MSE: 66.1528, Dq: 0.0000, KL: 18.83

...

[Epoch 73] Loss: 79.7743, FID: 204.2550, BCE: N/A
           MSE: 59.9105, Dq: 0.0000, KL: 19.8639, KL_t: 2.5517
[Epoch 74] Loss: 79.9204, FID: 203.3833, BCE: N/A
           MSE: 60.0671, Dq: 0.0000, KL: 19.8533, KL_t: 4.5454
[Epoch 75] Loss: 79.7521, FID: 202.0652, BCE: N/A
           MSE: 59.9081, Dq: 0.0000, KL: 19.8440, KL_t: 2.5367


Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-cy88mndo'


[Epoch 76] Loss: 80.0110, FID: 201.5831, BCE: N/A
           MSE: 60.1155, Dq: 0.0000, KL: 19.8955, KL_t: 1.8571
[Epoch 77] Loss: 79.6437, FID: 202.1777, BCE: N/A
           MSE: 59.7617, Dq: 0.0000, KL: 19.8820, KL_t: 1.8100
[Epoch 78] Loss: 80.0068, FID: 199.9745, BCE: N/A
           MSE: 60.1723, Dq: 0.0000, KL: 19.8345, KL_t: 2.1626
[Epoch 79] Loss: 79.7988, FID: 197.3424, BCE: N/A
           MSE: 59.9159, Dq: 0.0000, KL: 19.8829, KL_t: 1.6861
[Epoch 80] Loss: 79.8301, FID: 202.8604, BCE: N/A
           MSE: 59.9684, Dq: 0.0000, KL: 19.8617, KL_t: 2.6486
[Epoch 81] Loss: 79.8712, FID: 199.7989, BCE: N/A
           MSE: 59.9906, Dq: 0.0000, KL: 19.8806, KL_t: 1.7622
[Epoch 82] Loss: 79.8623, FID: 202.3711, BCE: N/A
           MSE: 59.9717, Dq: 0.0000, KL: 19.8906, KL_t: 2.0589
[Epoch 83] Loss: 79.6563, FID: 203.7158, BCE: N/A
           MSE: 59.7546, Dq: 0.0000, KL: 19.9017, KL_t: 1.8062


Traceback (most recent call last):
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 300, in _run_finalizers
    finalizer()
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 224, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/multiprocessing/util.py", line 133, in _remove_temp_dir
    rmtree(tempdir)
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 737, in rmtree
    onerror(os.rmdir, path, sys.exc_info())
  File "/home/yzm/anaconda3/envs/gen/lib/python3.11/shutil.py", line 735, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-n_w7evgj'


[Epoch 84] Loss: 79.6345, FID: 197.7889, BCE: N/A
           MSE: 59.7449, Dq: 0.0000, KL: 19.8896, KL_t: 2.1019
[Epoch 85] Loss: 79.6511, FID: 200.9348, BCE: N/A
           MSE: 59.7425, Dq: 0.0000, KL: 19.9086, KL_t: 2.5776
[Epoch 86] Loss: 79.7476, FID: 200.8324, BCE: N/A
           MSE: 59.8670, Dq: 0.0000, KL: 19.8806, KL_t: 2.1357
[Epoch 87] Loss: 79.6570, FID: 198.5316, BCE: N/A
           MSE: 59.7453, Dq: 0.0000, KL: 19.9116, KL_t: 2.4547
[Epoch 88] Loss: 79.6284, FID: 200.7889, BCE: N/A
           MSE: 59.7190, Dq: 0.0000, KL: 19.9094, KL_t: 1.7149
[Epoch 89] Loss: 79.6115, FID: 200.3693, BCE: N/A
           MSE: 59.7069, Dq: 0.0000, KL: 19.9046, KL_t: 1.9477
[Epoch 90] Loss: 79.6575, FID: 202.0927, BCE: N/A
           MSE: 59.7740, Dq: 0.0000, KL: 19.8835, KL_t: 2.0695
[Epoch 91] Loss: 79.5226, FID: 199.3324, BCE: N/A
           MSE: 59.6261, Dq: 0.0000, KL: 19.8965, KL_t: 1.9344
[Epoch 92] Loss: 79.3590, FID: 196.8865, BCE: N/A
           MSE: 59.4668, Dq: 0.0000, KL: 19.89

## eof